In [1]:
!pip install iterative-stratification tensorflow -q

import pandas as pd, numpy as np, pickle, ast
import matplotlib.pyplot as plt, seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.naive_bayes import ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (f1_score, precision_score, recall_score,
                              hamming_loss, jaccard_score, classification_report)
from scipy.special import expit

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

np.random.seed(42)
sns.set(style='whitegrid'); plt.rcParams['figure.dpi'] = 110
import warnings; warnings.filterwarnings('ignore')

You should consider upgrading via the 'd:\ta_220711657_baseline\venv\scripts\python.exe -m pip install --upgrade pip' command.


TypeError: unhashable type: 'list'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

SAVE_DIR = '/content/drive/MyDrive/skripsi_avoskin'

df = pd.read_csv(f'{SAVE_DIR}/df_clean.csv')
df['accept'] = df['accept'].apply(ast.literal_eval)

with open(f'{SAVE_DIR}/classes.pkl', 'rb') as f:
    classes = pickle.load(f)
with open(f'{SAVE_DIR}/fold_splits.pkl', 'rb') as f:
    fold_splits = pickle.load(f)

mlb = MultiLabelBinarizer(classes=classes)
texts = df["clean_text"].values
Y = mlb.fit_transform(df["accept"])

print(f"Data: {len(df)} baris, {len(classes)} label, {len(fold_splits)} fold")

In [ ]:
def evaluate_full(name, Yt, Yp, classes):
    agg = {
        'F1_micro'        : f1_score(Yt, Yp, average='micro', zero_division=0),
        'F1_macro'        : f1_score(Yt, Yp, average='macro', zero_division=0),
        'Precision_micro' : precision_score(Yt, Yp, average='micro', zero_division=0),
        'Precision_macro' : precision_score(Yt, Yp, average='macro', zero_division=0),
        'Recall_micro'    : recall_score(Yt, Yp, average='micro', zero_division=0),
        'Recall_macro'    : recall_score(Yt, Yp, average='macro', zero_division=0),
        'Hamming Loss'    : hamming_loss(Yt, Yp),
        'Jaccard (samples)': jaccard_score(Yt, Yp, average='samples', zero_division=0),
    }
    per_label = pd.DataFrame(
        classification_report(Yt, Yp, target_names=classes, output_dict=True, zero_division=0)
    ).T.loc[classes]
    return agg, per_label

def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return np.asarray(model.predict_proba(X))
    elif hasattr(model, "decision_function"):
        return expit(model.decision_function(X))
    raise ValueError("Model tidak punya predict_proba/decision_function")

def tune_threshold(model, Xval, Yval, grid=np.arange(0.1, 0.9, 0.05)):
    scores_val = get_scores(model, Xval)
    best_thr, best_f1 = 0.5, -1
    for thr in grid:
        f1 = f1_score(Yval, (scores_val > thr).astype(int), average='micro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_thr = f1, thr
    return best_thr, best_f1

def tune_threshold_per_label(model, Xval, Yval, grid=np.arange(0.1, 0.9, 0.05)):
    scores_val = get_scores(model, Xval)
    thr_list = []
    for j in range(Yval.shape[1]):
        best_thr, best_f1 = 0.5, -1
        for thr in grid:
            f1 = f1_score(Yval[:, j], (scores_val[:, j] > thr).astype(int), zero_division=0)
            if f1 > best_f1:
                best_f1, best_thr = f1, thr
        thr_list.append(best_thr)
    return np.array(thr_list)

def evaluate_threshold_variants(
    model, X_test, Y_test, X_val, Y_val,
    model_name, scenario, fold,
    all_agg, all_perlabel, all_thr, all_cm,
    final_preds, classes):
    """Menjalankan & mencatat 3 varian threshold (default/global/per-label) untuk 1 model di 1 fold."""
    thr_g, _   = tune_threshold(model, X_val, Y_val)
    thr_pl     = tune_threshold_per_label(model, X_val, Y_val)
    scores_test = get_scores(model, X_test)

    variants = {
        'threshold_default_0.5': (scores_test > 0.5).astype(int),
        'threshold_global'      : (scores_test > thr_g).astype(int),
        'threshold_per_label'   : (scores_test > thr_pl).astype(int),
    }

    for thr_mode, pred in variants.items():
        final_preds[f"{model_name}_{thr_mode}"] = (Y_test.copy(), pred.copy())
        agg, per_label = evaluate_full(f"{model_name}-{thr_mode}", Y_test, pred, classes)
        agg.update({'fold': fold, 'model': model_name, 'scenario': scenario, 'threshold_mode': thr_mode})
        all_agg.append(agg)

        per_label = per_label.reset_index().rename(columns={'index': 'label'})
        per_label['fold'] = fold; per_label['model'] = model_name
        per_label['scenario'] = scenario; per_label['threshold_mode'] = thr_mode
        all_perlabel.append(per_label)

        print(f"  {model_name:12s} | {thr_mode:20s} -> F1_micro={agg['F1_micro']:.4f}  F1_macro={agg['F1_macro']:.4f}")

    all_thr.append({'fold': fold, 'model': model_name, 'scenario': scenario,
                     'threshold_global': thr_g, 'threshold_per_label': list(thr_pl)})

In [ ]:
MAX_WORDS = 8000
MAX_LEN = 60

def build_models_tfidf():
    return {
        'Naive Bayes': OneVsRestClassifier(ComplementNB(alpha=0.5)),
        'SVM'        : OneVsRestClassifier(LinearSVC(C=1.0, max_iter=5000)),
    }

class LSTMWrapper:
    def __init__(self, keras_model):
        self.model = keras_model
    def predict_proba(self, X):
        return self.model.predict(X, verbose=0)

def build_lstm(n_labels, vocab_size):
    model = Sequential([
        Embedding(vocab_size, 128, input_length=MAX_LEN),
        (LSTM(64)),
        Dropout(0.3),
        Dense(64, activation='relu'),
        Dropout(0.3),
        Dense(n_labels, activation='sigmoid'),
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy')
    return model

In [ ]:
SKENARIO = 'murni'
all_agg, all_perlabel, all_thr, all_cm = [], [], [], []
final_preds = {}

for fold, (tr_full_idx, te_idx) in enumerate(fold_splits, start=1):
    print(f"\n{'='*25} FOLD {fold}/5 - {SKENARIO} {'='*25}")

    X_train_full_t, X_test_t = texts[tr_full_idx], texts[te_idx]
    Y_train_full, Y_test = Y[tr_full_idx], Y[te_idx]

    # --- Split train_full -> train-inti & validation, STRATIFIED ---
    msss = MultilabelStratifiedShuffleSplit(n_splits=1, test_size=0.125, random_state=42)
    tr_idx, val_idx = next(msss.split(X_train_full_t, Y_train_full))
    X_train_t, X_val_t = X_train_full_t[tr_idx], X_train_full_t[val_idx]
    Y_train, Y_val = Y_train_full[tr_idx], Y_train_full[val_idx]

    # --- TF-IDF: fit HANYA di train-inti ---
    tfidf = TfidfVectorizer(min_df=2, max_df=0.95, ngram_range=(1,2), sublinear_tf=True)
    X_train = tfidf.fit_transform(X_train_t)
    X_val   = tfidf.transform(X_val_t)
    X_test  = tfidf.transform(X_test_t)

    # --- Model NB, SVM ---
    for name, model in build_models_tfidf().items():
        model.fit(X_train, Y_train)
        evaluate_threshold_variants(model, X_test, Y_test, X_val, Y_val, name, SKENARIO, fold,
                                     all_agg, all_perlabel, all_thr, all_cm, final_preds, classes)

    # --- Tokenizer & padding KHUSUS LSTM, fit HANYA di train-inti ---
    tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token='<OOV>')
    tokenizer.fit_on_texts(X_train_t)

    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train_t), maxlen=MAX_LEN)
    X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val_t),   maxlen=MAX_LEN)
    X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test_t),  maxlen=MAX_LEN)

    vocab_size = min(MAX_WORDS, len(tokenizer.word_index) + 1)
    lstm_keras = build_lstm(n_labels=len(classes), vocab_size=vocab_size)
    history = lstm_keras.fit(X_train_seq, Y_train, epochs=15, batch_size=32,
                   validation_data=(X_val_seq, Y_val),
                   callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
                   verbose=0)

    lstm_model = LSTMWrapper(lstm_keras)
    evaluate_threshold_variants(lstm_model, X_test_seq, Y_test, X_val_seq, Y_val, 'LSTM', SKENARIO, fold,
                                 all_agg, all_perlabel, all_thr, all_cm, final_preds, classes)

    print(f"Fold {fold} selesai untuk semua model (NB, SVM, LSTM).")

df_agg_baseline = pd.DataFrame(all_agg)
df_perlabel_baseline = pd.concat(all_perlabel, ignore_index=True)
df_thr_baseline = pd.DataFrame(all_thr)

print("\nSemua fold selesai.")

In [ ]:
ringkasan_murni = (df_agg_baseline
    .groupby(['model', 'threshold_mode'])[['F1_micro','F1_macro','Precision_macro','Recall_macro', 'Hamming Loss','Jaccard (samples)']]
    .agg(['mean','std']).round(4))
display(ringkasan_murni)

In [ ]:
ringkasan_f1_perlabel_murni = (
    df_perlabel_baseline[df_perlabel_baseline['threshold_mode'] == 'threshold_per_label']
    .groupby(['model','label'])['f1-score']
    .agg(['mean','std'])
    .round(4)
    .reset_index()
)

display(ringkasan_f1_perlabel_murni)

In [ ]:
display(df_thr_baseline[['fold','model','threshold_global']])

print("\nContoh threshold per label (fold 1):")
display(pd.DataFrame(
    df_thr_baseline[df_thr_baseline.fold == 1]
    .set_index('model')['threshold_per_label']
    .apply(lambda lst: dict(zip(classes, np.round(lst, 2))))
    .to_dict()
))

In [ ]:
import pickle

with open('/content/drive/MyDrive/skripsi_avoskin/history_lstm_baseline.pkl', 'wb') as f:
    pickle.dump(history.history, f)

In [ ]:
import pickle
with open('/content/drive/MyDrive/skripsi_avoskin/result_baseline.pkl', 'wb') as f:
    pickle.dump(df_agg_baseline, f)

In [ ]:
from sklearn.metrics import multilabel_confusion_matrix

for name, (Y_true, pred) in final_preds.items():

    mcm = multilabel_confusion_matrix(Y_true, pred)

    fig, axes = plt.subplots(3,4, figsize=(18,12))

    for ax, cm, label in zip(axes.ravel(), mcm, classes):

        sns.heatmap(
            cm,
            annot=True,
            fmt='d',
            cmap='Blues',
            cbar=False,
            square=True,
            linewidths=0.5,
            ax=ax,
            xticklabels=['Neg','Pos'],
            yticklabels=['Neg','Pos']
        )

        ax.set_title(label, fontsize=10)
        ax.set_xlabel("Prediksi")
        ax.set_ylabel("Aktual")

    plt.suptitle(
        f"Confusion Matrix per Label — {name}",
        fontsize=16,
        y=1.02
    )

    plt.tight_layout()
    plt.show()

    print(f"\n===== {name} =====")
    print(classification_report(
        Y_true,
        pred,
        target_names=classes,
        zero_division=0
    ))

In [ ]:
# ============================================================
# CELL 11
# THRESHOLD FINAL SVM BASELINE PER LABEL DARI 5 FOLD
# ============================================================

import numpy as np
import pandas as pd

# ============================================================
# AMBIL HASIL THRESHOLD SVM BASELINE
# ============================================================

df_thr_svm_baseline = df_thr_baseline[
    (df_thr_baseline['model'] == 'SVM') &
    (df_thr_baseline['scenario'] == 'murni')
].copy()

print("Jumlah fold SVM baseline:", len(df_thr_svm_baseline))

# Pastikan benar-benar 5 fold
if len(df_thr_svm_baseline) != 5:
    raise ValueError(
        f"Threshold SVM baseline seharusnya berasal dari 5 fold, "
        f"tetapi ditemukan {len(df_thr_svm_baseline)}."
    )

# ============================================================
# AMBIL THRESHOLD PER LABEL
# ============================================================

threshold_matrix_baseline = np.array(
    df_thr_svm_baseline['threshold_per_label'].tolist(),
    dtype=float
)

print(
    "Shape threshold matrix:",
    threshold_matrix_baseline.shape
)

# ============================================================
# THRESHOLD FINAL
# ============================================================

threshold_per_label_final_baseline = threshold_matrix_baseline.mean(axis=0)

# Batasi agar tetap berada pada rentang 0.1 - 0.9
threshold_per_label_final_baseline = np.clip(
    threshold_per_label_final_baseline,
    0.1,
    0.9
)

print("\n==============================================")
print("THRESHOLD FINAL SVM BASELINE PER LABEL")
print("==============================================")

for label, threshold in zip(
    classes,
    threshold_per_label_final_baseline
):
    print(
        f"{label:25s} : {threshold:.4f}"
    )

# ============================================================
# CEK KHUSUS LABEL NEGATIF
# ============================================================

print("\n==============================================")
print("THRESHOLD LABEL NEGATIF")
print("==============================================")

for label, threshold in zip(
    classes,
    threshold_per_label_final_baseline
):
    if label.startswith("negatif_"):
        print(
            f"{label:25s} : {threshold:.4f}"
        )

In [ ]:
# ============================================================
# CELL 12
# TRAIN FINAL SVM BASELINE UNTUK DEPLOYMENT
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.multiclass import OneVsRestClassifier
from sklearn.svm import LinearSVC

# ============================================================
# 1. TF-IDF FINAL
# ============================================================

tfidf_final_baseline = TfidfVectorizer(
    min_df=2,
    max_df=0.95,
    ngram_range=(1, 2),
    sublinear_tf=True
)

# Fit TF-IDF menggunakan SELURUH DATA
X_full_baseline = tfidf_final_baseline.fit_transform(texts)

print(
    "Shape TF-IDF:",
    X_full_baseline.shape
)

# ============================================================
# 2. TRAIN SVM FINAL
# ============================================================

svm_final_baseline = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    )
)

svm_final_baseline.fit(
    X_full_baseline,
    Y
)

print("SVM baseline final berhasil dilatih.")

# ============================================================
# 3. CEK JUMLAH LABEL
# ============================================================

print(
    "Jumlah label model:",
    len(svm_final_baseline.estimators_)
)

print(
    "Jumlah classes:",
    len(classes)
)

In [ ]:
# ============================================================
# CELL 13
# SIMPAN FILE DEPLOYMENT SVM BASELINE
# ============================================================

import joblib
import pickle
import json
import os

# ============================================================
# 1. SIMPAN TF-IDF FINAL
# ============================================================

joblib.dump(
    tfidf_final_baseline,
    'preprocessor_tfidf_baseline.pkl'
)

print("✓ preprocessor_tfidf_baseline.pkl")


# ============================================================
# 2. SIMPAN SVM FINAL
# ============================================================

joblib.dump(
    svm_final_baseline,
    'svm_model_baseline.pkl'
)

print("✓ svm_model_baseline.pkl")


# ============================================================
# 3. SIMPAN THRESHOLD FINAL
# ============================================================

joblib.dump(
    threshold_per_label_final_baseline,
    'threshold_per_label_baseline.pkl'
)

print("✓ threshold_per_label_baseline.pkl")


# ============================================================
# 4. SIMPAN CLASSES
# ============================================================

with open(
    'classes_baseline.pkl',
    'wb'
) as f:

    pickle.dump(
        classes,
        f
    )

print("✓ classes_baseline.pkl")


# ============================================================
# 5. SIMPAN HASIL EVALUASI
# ============================================================

best_result_baseline = ringkasan_murni.loc[
    ('SVM', 'threshold_per_label')
]

results_baseline = {}

for key, value in best_result_baseline.items():

    if isinstance(key, tuple):
        key = '_'.join(str(x) for x in key)

    if pd.notna(value):
        results_baseline[str(key)] = float(value)


# Tambahkan threshold yang digunakan
results_baseline["threshold_per_label"] = {
    str(label): float(threshold)
    for label, threshold in zip(
        classes,
        threshold_per_label_final_baseline
    )
}


with open(
    'svm_results_baseline.json',
    'w'
) as f:

    json.dump(
        results_baseline,
        f,
        indent=4
    )

print("✓ svm_results_baseline.json")


# ============================================================
# 6. SIMPAN RINGKASAN MODEL
# ============================================================

ringkasan_murni.to_csv(
    'model_summary_baseline.csv',
    index=True
)

print("✓ model_summary_baseline.csv")


# ============================================================
# 7. CEK FILE
# ============================================================

print("\n==============================================")
print("FILE DEPLOYMENT SVM BASELINE")
print("==============================================")

files_baseline = [
    'preprocessor_tfidf_baseline.pkl',
    'svm_model_baseline.pkl',
    'threshold_per_label_baseline.pkl',
    'classes_baseline.pkl',
    'svm_results_baseline.json',
    'model_summary_baseline.csv'
]

for file in files_baseline:

    if os.path.exists(file):

        size = os.path.getsize(file) / 1024

        print(
            f"✓ {file} ({size:.1f} KB)"
        )

    else:

        print(
            f"✗ {file} TIDAK DITEMUKAN"
        )